# Session 4: Text Embeddings & Semantic Search

## Objectives
- Understand what text embeddings are and why they matter
- Generate embeddings using the OpenAI API
- Compute similarity between texts
- Build a semantic search engine from scratch

**Duration:** 40 minutes | **Level:** Medium

**Why this matters:** Embeddings are the foundation of RAG, recommendation systems, and similarity-based applications.

In [ ]:
!pip install openai numpy python-dotenv -q

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
load_dotenv(dotenv_path=os.path.join("..", ".env"))

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

print(f"Setup complete! Model: {MODEL}")

## 1. What Are Embeddings?

An **embedding** is a vector (list of numbers) that represents the *meaning* of text.

- Similar texts â†’ vectors that are close together
- Different texts â†’ vectors that are far apart
- Dimensions capture semantic features (topic, sentiment, style, etc.)

```
"I love dogs"     â†’ [0.12, -0.34, 0.56, ...] (1536 dimensions)
"I adore puppies"  â†’ [0.11, -0.33, 0.57, ...] (very similar!)
"Quantum physics"  â†’ [-0.45, 0.78, -0.12, ...] (very different!)
```

In [ ]:
# Generate an embedding for a single text
response = client.embeddings.create(
    model="text-embedding-3-small",  # Fast and cost-effective
    input="Machine learning is a subset of artificial intelligence."
)

embedding = response.data[0].embedding

print(f"Embedding dimensions: {len(embedding)}")
print(f"First 10 values: {embedding[:10]}")
print(f"Type: {type(embedding)}")

In [ ]:
# Helper function to get embeddings
def get_embedding(text, model="text-embedding-3-small"):
    """Get the embedding vector for a text string."""
    response = client.embeddings.create(model=model, input=text)
    return response.data[0].embedding

# Get embeddings for multiple texts at once (more efficient)
def get_embeddings(texts, model="text-embedding-3-small"):
    """Get embeddings for a list of texts in a single API call."""
    response = client.embeddings.create(model=model, input=texts)
    return [item.embedding for item in response.data]

print("Helper functions defined!")

## 2. Measuring Similarity with Cosine Similarity

**Cosine similarity** measures the angle between two vectors:
- `1.0` = identical meaning
- `0.0` = unrelated
- `-1.0` = opposite meaning

Formula: $\cos(\theta) = \frac{A \cdot B}{\|A\| \|B\|}$

In [ ]:
def cosine_similarity(vec_a, vec_b):
    """Compute cosine similarity between two vectors."""
    a = np.array(vec_a)
    b = np.array(vec_b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Compare similar and different texts
texts = [
    "I love programming in Python",       # Text 0
    "Python coding is my favorite hobby",  # Text 1 (similar to 0)
    "The weather is sunny today",          # Text 2 (different topic)
]

embeddings = get_embeddings(texts)

# Compare all pairs
print("Similarity scores:")
print(f"  'programming in Python' vs 'Python coding': {cosine_similarity(embeddings[0], embeddings[1]):.4f}")
print(f"  'programming in Python' vs 'sunny weather': {cosine_similarity(embeddings[0], embeddings[2]):.4f}")
print(f"  'Python coding' vs 'sunny weather':         {cosine_similarity(embeddings[1], embeddings[2]):.4f}")

## 3. Building a Semantic Search Engine

The idea is simple:
1. **Index**: Embed all your documents
2. **Search**: Embed the query, find the closest documents
3. **Return**: Return the most similar documents

In [ ]:
# Our knowledge base â€” a collection of facts
knowledge_base = [
    "Python was created by Guido van Rossum and released in 1991.",
    "Machine learning models learn patterns from data without being explicitly programmed.",
    "The Transformer architecture was introduced in the 'Attention Is All You Need' paper in 2017.",
    "GPT stands for Generative Pre-trained Transformer.",
    "Neural networks are inspired by the structure of biological neurons in the brain.",
    "Docker containers package applications with their dependencies for consistent deployment.",
    "REST APIs use HTTP methods like GET, POST, PUT, DELETE for communication.",
    "PostgreSQL is an open-source relational database management system.",
    "Large Language Models are trained on massive amounts of text data from the internet.",
    "Fine-tuning adapts a pre-trained model to a specific task using domain-specific data."
]

# Step 1: Embed all documents (this is the "indexing" step)
kb_embeddings = get_embeddings(knowledge_base)
print(f"Indexed {len(knowledge_base)} documents")
print(f"Each embedding has {len(kb_embeddings[0])} dimensions")

In [ ]:
def semantic_search(query, documents, doc_embeddings, top_k=3):
    """Search for the most relevant documents given a query."""
    # Step 2: Embed the query
    query_embedding = get_embedding(query)
    
    # Step 3: Compute similarity with all documents
    similarities = [
        cosine_similarity(query_embedding, doc_emb)
        for doc_emb in doc_embeddings
    ]
    
    # Step 4: Sort by similarity and return top results
    scored_docs = list(zip(similarities, documents))
    scored_docs.sort(key=lambda x: x[0], reverse=True)
    
    return scored_docs[:top_k]

# Test the search engine
query = "How do transformers work?"
results = semantic_search(query, knowledge_base, kb_embeddings)

print(f"Query: '{query}'\n")
for score, doc in results:
    print(f"  [{score:.4f}] {doc}")

In [ ]:
# Try different queries â€” notice how semantic search understands meaning!
queries = [
    "What programming language was made by Guido?",
    "How to deploy applications reliably?",
    "How are AI models trained?"
]

for query in queries:
    results = semantic_search(query, knowledge_base, kb_embeddings, top_k=2)
    print(f"\nQuery: '{query}'")
    for score, doc in results:
        print(f"  [{score:.4f}] {doc}")

## 4. Semantic Search vs Keyword Search

Semantic search finds results based on **meaning**, not just matching keywords.

In [ ]:
# Keyword search (simple but limited)
def keyword_search(query, documents, top_k=3):
    """Simple keyword matching search."""
    query_words = set(query.lower().split())
    scored = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words)
        scored.append((overlap, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

# This query uses different words than any document
query = "How do neural networks learn?"

print("=== Keyword Search ===")
for score, doc in keyword_search(query, knowledge_base):
    print(f"  [matches: {score}] {doc}")

print("\n=== Semantic Search ===")
for score, doc in semantic_search(query, knowledge_base, kb_embeddings):
    print(f"  [{score:.4f}] {doc}")

## Exercise: FAQ Search System

Build a FAQ search system that finds the most relevant answer to a user's question.

In [ ]:
# FAQ database
faqs = [
    {"q": "How do I reset my password?", "a": "Go to Settings > Security > Reset Password and follow the instructions."},
    {"q": "What payment methods do you accept?", "a": "We accept Visa, MasterCard, PayPal, and bank transfers."},
    {"q": "How can I cancel my subscription?", "a": "Navigate to Account > Subscription > Cancel. Your access continues until the end of the billing period."},
    {"q": "Do you offer a free trial?", "a": "Yes! We offer a 14-day free trial with full access to all features."},
    {"q": "How do I contact support?", "a": "You can reach us via email at support@example.com or use the live chat on our website."},
    {"q": "What is your refund policy?", "a": "We offer a 30-day money-back guarantee. Contact support for refunds."},
]

# Embed all FAQ questions
faq_questions = [faq["q"] for faq in faqs]
faq_embeddings = get_embeddings(faq_questions)

def search_faq(user_question):
    """Find the most relevant FAQ answer."""
    results = semantic_search(user_question, faq_questions, faq_embeddings, top_k=1)
    best_score, best_question = results[0]
    # Find the matching FAQ
    for faq in faqs:
        if faq["q"] == best_question:
            return faq["a"], best_score

# Test with user questions (phrased differently than the FAQs!)
test_questions = [
    "I forgot my login credentials",
    "Can I pay with PayPal?",
    "I want to stop my subscription",
    "Can I try before buying?"
]

for q in test_questions:
    answer, score = search_faq(q)
    print(f"Q: {q}")
    print(f"A: {answer} (confidence: {score:.4f})")
    print()

## Summary

**What you learned:**
- Embeddings convert text into numerical vectors capturing meaning
- Cosine similarity measures how similar two texts are
- Semantic search finds relevant results even with different wording
- The index-then-search pattern is the foundation of many AI applications

**Next session:** We'll combine embeddings with LLM generation to build RAG (Retrieval-Augmented Generation)!